# L3d: Comparing Sorting Algorithms and Their Scaling
In this lab, we compare the average runtime of a recursive [Quicksort](https://en.wikipedia.org/wiki/Quicksort) implementation against a [Bubblesort implementation](https://en.wikipedia.org/wiki/Bubble_sort) and [Julia's built-in sort function](https://docs.julialang.org/en/v1/base/sort/#Base.sort) using [the BenchmarkTools.jl package](https://github.com/JuliaCI/BenchmarkTools.jl).

> __Learning Objectives:__
>
> By the end of this lab, you should be able to:
> * __Test an implementation before timing it:__ Establish that a sorting routine reproduces the reference ordering before measuring how fast it runs, because a quick wrong answer is worth nothing.
> * __Measure how runtime scales:__ Benchmark one operation across growing inputs and read the resulting curve, separating the growth rate from the constant factor sitting in front of it.
> * __Decide between building and buying:__ Compare a transparent teaching implementation against the library routine, and say what the library is actually buying you beyond asymptotic behavior.

We'll complete three main tasks in this notebook:
- **Task 1**: Verify [our `bubblesort(...)` implementation](src/Compute.jl) against [Julia's `sort(...)` function](https://docs.julialang.org/en/v1/base/sort/#Base.sort), then benchmark it across array sizes to establish a baseline for the other algorithms.
- **Task 2**: Verify [our `quicksort(...)` implementation](src/Compute.jl) the same way and benchmark it against that baseline.
- **Task 3**: Benchmark Julia's built-in sort to see how both custom implementations compare against an optimized library routine.

### Algorithms
Bubblesort is a simple method of ordering a list. 

> __How does Bubblesort work?__ It involves repeatedly passing through the list, comparing adjacent items, and swapping any two neighboring items that are out of order. The algorithm gets its name because smaller elements "bubble" to the top of the list, just like how air bubbles rise to the surface of water.
> 
> Let's take a look at the Bubblesort algorithm in more detail [here](CHEME-5800-L3d-Algorithm-Bubblesort-Fall-2026.ipynb).


Quicksort takes a different approach to sorting. 

> __How does Quicksort work?__ Quicksort is a recursive sorting algorithm that works by selecting a pivot element and partitioning the remaining elements according to their value relative to the pivot. The algorithm then recursively sorts the partitions until they have fewer than two elements. The choice of pivot is critical for the algorithm's efficiency.
> 
> Let's take a look at the Quicksort algorithm [here](CHEME-5800-L3d-Algorithm-Quicksort-Fall-2026.ipynb).

We are going to see some surprising results! Let's get started!
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> The [include command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 

Let's set up our environment:

In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # what is this doing?

The course environment also loads [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl); see [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/). This lab does not need it. The sorting implementations we benchmark live in [`src/Compute.jl`](src/Compute.jl), and the timing comes from [the BenchmarkTools.jl package](https://github.com/JuliaCI/BenchmarkTools.jl).

### Constants
Before we get started, let's set up some constants. See the comment next to each value for what it is, its permissible values, units, etc.

In [ ]:
max_number_of_trials = 10; # exponent; largest vector has 2^10 elements
number_of_items_per_trial = [2^i for i ∈ 1:max_number_of_trials]; # this is an array comprehension, yet another iteration pattern!

___

## Task 1: Establish a performance baseline: Bubblesort
In this task, we show that our bubblesort implementation works as expected. We also establish a performance baseline for comparison with the Quicksort algorithm.

> We'll use [the `Test.jl` package](https://docs.julialang.org/en/v1/stdlib/Test/) to write __unit tests__ for our bubble sort implementation. The [`Test.jl` package](https://docs.julialang.org/en/v1/stdlib/Test/) provides a framework for writing and running tests in Julia, including support for assertions, test cases, and test suites. Let's use [the @test macro](https://docs.julialang.org/en/v1/stdlib/Test/#Test.@test) to check that our bubble sort implementation works correctly.

So, did we pass the tests?

In [ ]:
let

    # initialize -
    N = 1000; # number of elements in the random vector
    arr = rand(N); # random vector length N

    # check: do we get the same result as the built-in sort function?
    @test sort(arr) == bubblesort(arr) # if the test fails, an error is thrown!
end

Okay, so if we get here, all seems to be good with [our `bubblesort(...)` implementation](src/Compute.jl), so let's see how our code performs as we increase the size of the vector that we are sorting.

Now let's measure how [our `bubblesort(...)` function](src/Compute.jl) performs as we increase the size of the input array. We'll use the [BenchmarkTools.jl package](https://github.com/JuliaCI/BenchmarkTools.jl) to get accurate timing measurements.

The code below uses a [`let ... end` block](https://docs.julialang.org/en/v1/manual/variables-and-scoping/#Let-Blocks) to create a local scope and performs the following steps:
1. **Initialize a DataFrame** to store our results with columns for array size (`n`), mean runtime (`μ`), and standard deviation (`σ`) using the [DataFrames.jl package](https://github.com/JuliaData/DataFrames.jl)
2. **Loop through different array sizes** from our `number_of_items_per_trial` array (powers of 2 from $2^{1}$ to $2^{\texttt{max\_number\_of\_trials}}$, which is $2^{10}$ as set above)
3. **For each array size:**
   - Create a benchmark using [the `@benchmarkable` macro](https://juliaci.github.io/BenchmarkTools.jl/stable/reference/#BenchmarkTools.@benchmarkable-Tuple) that will test our `bubblesort` function
   - Use `setup=` to generate fresh random data for each trial (avoiding measuring data generation time)
   - Run the benchmark multiple times and collect timing statistics
   - Store the results (array size, mean time, standard deviation) as a row in our DataFrame

The result will be stored in the `bubble_sort_data::DataFrame` variable containing performance data that we can analyze and visualize.

In [ ]:
bubble_sort_data = let
    bubble_sort_data = DataFrame();
    for i ∈ eachindex(number_of_items_per_trial)
        size_of_rand_vec_to_sort = number_of_items_per_trial[i];
    
        # run the test with different size vectors -
        test_run = @benchmarkable bubblesort(data) setup=(data=rand(0:number_of_items_per_trial[end], $(size_of_rand_vec_to_sort)));
        results = run(test_run; samples = 20, seconds = 0.25, evals = 1)
    
        # store the results -
        row = (
            n = size_of_rand_vec_to_sort,
            μ = mean(results.times),
            σ = std(results.times)
        );
        push!(bubble_sort_data, row)
    end
    bubble_sort_data
end

## Task 2: Quicksort
In this task, we verify that our Quicksort implementation works as expected. We also establish the performance of this method in comparison with the Bubblesort algorithm.

> As before, we'll use [the `Test.jl` package](https://docs.julialang.org/en/v1/stdlib/Test/) to write a __unit test__ for [our `quicksort(...)` implementation](src/Compute.jl). The [`Test.jl` package](https://docs.julialang.org/en/v1/stdlib/Test/) provides a framework for writing and running tests in Julia, including support for assertions, test cases, and test suites. Let's use [the @test macro](https://docs.julialang.org/en/v1/stdlib/Test/#Test.@test) to check that [our `quicksort(...)` implementation](src/Compute.jl) works correctly.

Does it work?

In [ ]:
let

    # initialize -
    N = 1000; # number of elements in the random vector
    arr = rand(N); # random vector length N

    # check: do we get the same result as the built-in sort function?
    @test sort(arr) == quicksort(arr) # if the test fails, an error is thrown!
end

Okay, so if we get here, all seems to be good with our `quicksort(...)` implementation, so let's see how our code performs as we increase the vector size to be sorted. 

Let's use the same benchmarking pattern as we did for the Bubblesort algorithm. The benchmarking results for the Quicksort algorithm will be stored in the `quick_sort_data::DataFrame` variable containing performance data that we can analyze and visualize.

In [ ]:
quick_sort_data = let
    
    quick_sort_data = DataFrame();
    for i ∈ eachindex(number_of_items_per_trial)
        size_of_rand_vec_to_sort = number_of_items_per_trial[i];
    
        # run the test with different size vectors -
        test_run = @benchmarkable quicksort(data) setup=(data=rand(0:number_of_items_per_trial[end], $(size_of_rand_vec_to_sort)));
        results = run(test_run; samples = 20, seconds = 0.25, evals = 1)
    
        # store the results -
        row = (
            n = size_of_rand_vec_to_sort,
            μ = mean(results.times),
            σ = std(results.times)
        );
        push!(quick_sort_data, row)
    end
    quick_sort_data
end

## Task 3: What is the scaling of the Built-in sort function?
Julia provides [sophisticated multi-method sorting capability](https://docs.julialang.org/en/v1/base/sort/#Sorting-Functions). How do our implementations perform against what Julia can offer? 

> __Buy versus build:__ This is yet another example of the __buy versus build__ conundrum. Should we build our own implementation, or __buy__ someone else's? You should (almost) always __buy__, and benefit from the hard (optimized) work of others. But let's see if that is true in this case.

We'll use the same benchmarking approach as we did for the Bubblesort and Quicksort algorithms. The results will be stored in the `julia_sort_data::DataFrame` variable containing performance data that we can analyze and visualize.

In [ ]:
julia_sort_data = let
    julia_sort_data = DataFrame();
    for i ∈ eachindex(number_of_items_per_trial)
        size_of_rand_vec_to_sort = number_of_items_per_trial[i];
    
        # run the test with different size vectors -
        test_run = @benchmarkable sort(data) setup=(data=rand(0:number_of_items_per_trial[end], $(size_of_rand_vec_to_sort)));
        results = run(test_run; samples = 20, seconds = 0.25, evals = 1)
    
        # store the results -
        row = (
            n = size_of_rand_vec_to_sort,
            μ = mean(results.times),
            σ = std(results.times)
        );
        push!(julia_sort_data, row)
    end
    julia_sort_data
end

## Visualize
Unhide the code to see how we plotted the average runtime of each sorting method as a function of the length of the vector $n$.

> __What the curves show__: For very short sequences, our `bubblesort(...)` implementation is the winner! However, once the sequences become large, [Julia `sort(...)` implementations are the clear winners](https://docs.julialang.org/en/v1/base/sort/#Sorting-Functions). Something interesting here: our `quicksort(...)` implementation seems to have similar scaling behavior to [the built-in `sort(...)` method](https://docs.julialang.org/en/v1/base/sort/#Sorting-Functions).

These results show the power of constants! When we do scaling analysis, we ignore constants, but they are important in practice. For example, Quicksort and Julia's built-in sort function have similar scaling behavior, but the constants are different (the built-in sort implementation appears to have a smaller constant factor, meaning it is faster in practice).

In [ ]:
let
    plot(quick_sort_data[:,:n], quick_sort_data[:,:μ], label="quicksort", 
        yscale=:log10, xscale=:log10, lw=3, c=:gray69, minorgrid=true, legend=:topleft)
    plot!(bubble_sort_data[:,:n], bubble_sort_data[:,:μ], label="bubblesort", 
        yscale=:log10, xscale=:log10, lw=3, c=:red)
    plot!(julia_sort_data[:,:n], julia_sort_data[:,:μ], label="Julia sort", 
        yscale=:log10, xscale=:log10, lw=3, c=:blue)
    xlims!(1e+0, 1.5e+3) # data runs to 2^10 = 1024
    ylims!(1e+0, 1e+7)
    xlabel!("Number of elements n", fontsize=18)
    ylabel!("Mean Runtime (ns)", fontsize=18)
end

## Summary
Sorting the same data three ways, and timing each across growing inputs, separates how an algorithm scales from how fast it actually runs.

> __Key Takeaways:__
>
> * **Correctness comes before speed:** An implementation is worth benchmarking only once it reproduces the reference ordering, which is why each task above tests before it times.
> * **Growth rate and constant factor are separate claims:** Two implementations can share a scaling exponent and still differ by a large multiple, so curves that look parallel on a log-log plot are not a statement that the two are equally fast.
> * **Buying usually beats building, for reasons worth knowing:** [Julia's `sort(...)` function](https://docs.julialang.org/en/v1/base/sort/#Base.sort) outruns our quicksort not by using a better asymptotic algorithm but through a great deal of engineering underneath it, which is the ordinary argument for reaching for a library rather than writing your own.

Our bubblesort wins for very small inputs and then loses badly as the data grows, which is what an $\mathcal{O}(n^2)$ method looks like when you plot it. Implementing an algorithm yourself is the way to understand it; production code should still call the library.
___